In [1]:
### Import Libraries
import pandas as pd
from gensim.models import Word2Vec
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from model_helper import *
from tqdm import tqdm
from sklearn.metrics import f1_score
import gc

In [2]:
### Load Data:
word_model_path = #path word-emb.model 
code_model_path = #path code-emb.model
x_path = #path train_word-emb.pt
y_path = #path train_code-emb.pt
word_model = Word2Vec.load(word_model_path)
code_model = Word2Vec.load(code_model_path)
x = torch.load(x_path, weights_only = True)
y = torch.load(y_path, weights_only = True)

In [3]:
### Hyperparameters:
### Define hidden size and embedding dim
hidden_size = 100
embed_dim = 100
batch_size = 32
num_sent = 25
lr = 0.01
dropout_prob = 0.5
l2 = 0.0001

calibration = 0.5

### Define dimension and padding for the embedding layer loaded from pre-trained model
dim = 100
pad = 0
padded_vec = np.zeros(dim)
padding_token = "<PAD>"

### Dropout prob
dropout_prob = 0.2

word_to_index, word_weight_tensor = get_word_index_wgts(word_model, padding_token, dim, padded_vector = padded_vec)
code_weight_tensor = torch.tensor(code_model.wv.vectors)

### Define the model

In [4]:
### Torch model:
class HLAN(nn.Module):
    def __init__(self, num_sent, word_weight_tensor, code_weight_tensor, embed_dim, hidden_size,dropout_prob):
        super(HLAN, self).__init__()
        self.num_sent = num_sent
        self.embed_dim = embed_dim
        self.hidden_size = hidden_size
        self.dropout_prob = dropout_prob

        self.embed_layer = nn.Embedding.from_pretrained(word_weight_tensor)
        self.gru_w = nn.GRU(input_size = embed_dim, hidden_size = hidden_size, bidirectional=True, batch_first=True)
        self.W_w = nn.Linear(hidden_size*2, hidden_size*2)
        self.attn_tanh = nn.Tanh()
        self.V_w_l = nn.Parameter(torch.randn(50, hidden_size*2))
        self.word_softmax = nn.Softmax(dim = 2)

        self.gru_s = nn.GRU(input_size = hidden_size * 2, hidden_size = hidden_size * 2, bidirectional = True, batch_first = True)
        self.W_s = nn.Linear(hidden_size * 4, hidden_size * 2)
        self.V_s_l = nn.Parameter(torch.randn(50, hidden_size *2))
        self.drop_layer = nn.Dropout(p = dropout_prob)
        self.sentence_softmax = nn.Softmax(dim =2)

        self.W_projection = nn.Parameter(code_weight_tensor.clone())
        self.sigmoid_act = nn.Sigmoid()

    def forward(self, batch):
        embed_comp = self.embed_layer(batch)

        embed_comp_reshape = embed_comp.view(-1, embed_comp.shape[2], embed_comp.shape[3])

        C_s_l = self.word_attention(embed_comp_reshape)

        doc_rep = self.sentence_attention(C_s_l)

        logits = self.apply_final_layer(doc_rep)

        probs = self.sigmoid_act(logits)

        return probs


    def word_attention(self, X):
        hidden_state, hnn = self.gru_w(X)
        hidden_state_reshape = hidden_state.reshape(-1,hidden_state.size(-1))
        hidden_rep_step = self.attn_tanh(self.W_w(hidden_state_reshape))
        v = hidden_rep_step.reshape(-1, hidden_state.size(1),hidden_state.size(-1))

        a_w_l = torch.matmul(v,self.V_w_l.T).view(-1,v.size(0),v.size(1))
        a_w_l = self.word_softmax(a_w_l).unsqueeze(3)
        C_s_l = a_w_l*hidden_state
        C_s_l = torch.sum(C_s_l,dim = 2)

        return C_s_l

    def sentence_attention(self, X):
        X.reshape = X.permute(1, 0 , 2)
        S_l, hnn = self.gru_s(X.reshape)

        hidden_rep_step = self.attn_tanh(self.W_s(S_l)) 
        U = hidden_rep_step.reshape(hidden_rep_step.size(1), -1, self.num_sent, hidden_rep_step.size(-1))

        S_l_reshape = S_l.reshape(S_l.size(1), -1, self.num_sent, S_l.size(2))
        V_s_l_expand = self.V_s_l.unsqueeze(1).unsqueeze(1)
        attention_logits = (U * V_s_l_expand).sum(dim=3)
        p_attention_sent = self.sentence_softmax(attention_logits - attention_logits.max(dim=2, keepdim=True).values)
        document_representation = (p_attention_sent.unsqueeze(3) * S_l_reshape).sum(dim=2)
        
        return document_representation

    def apply_final_layer(self, doc_rep):
        drop_step = self.drop_layer(doc_rep) 
        drop_step_reshape = drop_step.permute(1,2,0)
        logits = drop_step_reshape * self.W_projection.T
        logits = logits.sum(dim = 1)

        return logits


### Define training code

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
model = HLAN(num_sent = num_sent, word_weight_tensor=word_weight_tensor,code_weight_tensor=code_weight_tensor,embed_dim=embed_dim,hidden_size=hidden_size,dropout_prob=dropout_prob)
model.to(device)

HLAN(
  (embed_layer): Embedding(150855, 100)
  (gru_w): GRU(100, 100, batch_first=True, bidirectional=True)
  (W_w): Linear(in_features=200, out_features=200, bias=True)
  (attn_tanh): Tanh()
  (word_softmax): Softmax(dim=2)
  (gru_s): GRU(200, 200, batch_first=True, bidirectional=True)
  (W_s): Linear(in_features=400, out_features=200, bias=True)
  (drop_layer): Dropout(p=0.2, inplace=False)
  (sentence_softmax): Softmax(dim=2)
  (sigmoid_act): Sigmoid()
)

In [7]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = lr, weight_decay = l2)
num_epochs = 2

In [ ]:
dataset = TensorDataset(x, y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

best_micro_f1 = 0.0

# Run model epochs
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    for inputs, targets in tqdm(dataloader):
        inputs, targets = inputs.to(device), targets.float().to(device)

        optimizer.zero_grad()  

        outputs = model(inputs)
        loss = loss_fn(outputs, targets)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()  

        del inputs, targets, outputs, loss
        if str(device).startswith('cuda'):
            torch.cuda.empty_cache()
            gc.collect()

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss / len(dataloader):.4f}')

    model_save_path = f'model_epoch_{epoch + 1}.pth'
    torch.save(model.state_dict(), model_save_path)

    if str(device).startswith('cuda'):
        torch.cuda.empty_cache()
        gc.collect()

    model.load_state_dict(torch.load(model_save_path,weights_only = True))
    model.to(device)

if str(device).startswith('cuda'):
    torch.cuda.empty_cache()
    gc.collect()